In [ ]:
import sys
import os
import random
import time
from pathlib import Path

import torch
import torchvision
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

In [ ]:
project_root = Path.cwd().parent 
sys.path.append(str(project_root))

In [ ]:
from img_generation.datasets import Config, DatasetType, load_dataloaders, VAE_TRANSFORM_MONO
from img_generation.models.vae import VAE, train_vae, vae_loss_function
from img_generation.utils import EarlyStopping

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
device

In [ ]:
config = Config(dataset_type=DatasetType.FASHION_MNIST, data_dir="../data/fashion_mnist", model_type="vae", batch_size=128, num_workers=4)
dataloaders, classes = load_dataloaders(config=config, transform=VAE_TRANSFORM_MONO)

print(f"All classes: {classes}")    # tiny-imagenet labels are of poor quality
print(f"Number of all classes: {len(classes)}")

In [ ]:
for x, y in dataloaders["train"]:
    print(f"Input img shape: {x.shape}")
    print(f"class: {y}")
    break

In [ ]:
for x, _ in dataloaders["train"]:
    input_channels, input_size = x.shape[1], x.shape[2] # (B, C, H, W)
    break

In [ ]:
latent_dim = 64
hidden_dim = 32
y_embed_dim = 32


vae = VAE(input_channels=input_channels, hidden_dim_channels=hidden_dim, latent_dim=latent_dim, num_classes=len(classes), class_embed_dim=y_embed_dim).to(device=device)
optim = torch.optim.Adam(vae.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optim, gamma=0.99)
early_stopping = EarlyStopping(patience=10, delta = 0.1)

In [ ]:
epochs = 70
warmup_epochs = 50
max_beta = 0.5
# max_beta = 1.0

start = time.time()
early_stopping, val_lossses = train_vae(model=vae, loaders=dataloaders, criterion=vae_loss_function, optimizer=optim, lr_scheduler=scheduler, 
                                early_stopping=early_stopping, num_epochs=epochs, warmup_epochs=warmup_epochs, loss_beta=max_beta, device=device)
end = time.time()
early_stopping.load_best_model(vae)

plt.plot(val_lossses)

In [ ]:
model_path = "../pretrained_models/vae"

torch.save(vae.state_dict(), f=os.path.join([model_path, "cvae.pth"]))

In [ ]:
training_time = (end - start) / 60
print(f"Training lasted for {training_time} minutes")

In [ ]:
def visualize_reconstructions(model, input_imgs, labels, device):
    # Reconstruct images
    model.eval()
    with torch.no_grad():
        reconst_imgs, means, log_var = model(input_imgs.to(device), labels.to(device))
    reconst_imgs = reconst_imgs.cpu()
    
    # Plotting
    imgs = torch.stack([input_imgs, reconst_imgs], dim=1).flatten(0,1)
    grid = torchvision.utils.make_grid(imgs, nrow=6, normalize=False)
    grid = grid.permute(1, 2, 0)
    if len(input_imgs) == 6:
        plt.figure(figsize=(15,10))
    else:
        plt.figure(figsize=(15,15))
    plt.title("Reconstructions")
    plt.imshow(grid)
    plt.axis('off')
    plt.show()

In [ ]:
vae.eval()
with torch.inference_mode():    
    test_losses = []
    for x, y in tqdm(dataloaders["test"]):
        x = x.to(device)
        y = y.to(device)
        x_hat, mean, logvar = vae(x, y)
        total_loss, _ = vae_loss_function(x, x_hat, mean, logvar)
        test_losses.append(total_loss.item())
    print(f"Mean test loss: {np.mean(np.array(test_losses))}")

In [ ]:
n_samples = 27
sample_imgs = torch.stack([dataloaders["test"].dataset[i][0] for i in range(n_samples)], dim=0)    # wraps all tensors in one one external tensor - [[..], [..], .., [..]]
sample_lables = torch.tensor([dataloaders["test"].dataset[i][1] for i in range(n_samples)])
print(sample_imgs.shape)
print(sample_lables.shape)
visualize_reconstructions(vae, input_imgs=sample_imgs, labels=sample_lables, device=device)

### Generations form noise

In [ ]:
num_samples = 20

vae.eval()
with torch.no_grad():
    fixed_noise = torch.randn(num_samples, latent_dim, device=device)
    rnd_labels = torch.randint(0, len(classes), size=(num_samples,), device=device)
    generations_vae = vae.decoder(fixed_noise, rnd_labels)
    

In [ ]:
grid = torchvision.utils.make_grid(generations_vae.cpu()[:20], nrow=4, normalize=False) # produces 'channels first' format
grid = grid.permute(1, 2, 0)    # matplotplit.imshow takes 'channels' last format
if len(generations_vae) == 4:
    plt.figure(figsize=(15,10))
else:
    plt.figure(figsize=(15,15))
plt.title("Generations from noise")
plt.imshow(grid)
plt.axis('off')
plt.show()